In [1]:
# epoch影响较大
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --n_epochs 150 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 100 \
    --latent_dim 256 \
    --output_dir ../deconv_results/PDAC \
    --aggregation_method mean \
    --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 2.0
   Precomputed marker genes: /mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes.txt
   Batch size: 256
   Epochs: 150
   Learning rate: 0.001
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC.h5ad
   SC shape: (1926, 19738)
   SC avg counts/cell: 3302.8 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad
   ST shape: (428, 19738)
   Common genes: 19738
   SC final: (1926, 19738)
   ST final: (428, 19738)
Using precomputed marker genes from: /mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes.txt
Loading marker genes from file: /mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes

VAE Training:  90%|█████████ | 135/150 [00:42<00:04,  3.17epoch/s, Train=1228.1696, Recon=1221.9869, KL=61.8252, MMD=0.0220, Test=1296.6908]



⚠️ Early stopping triggered at epoch 136/150
   Best test loss: 1290.2948, Current test loss: 1296.6908
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (1926, 915)
   Number of clusters: 27
   Computed embeddings shape: (1926, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 193 cells (mean aggregation)
      Cluster 1: 160 cells (mean aggregation)
      Cluster 2: 85 cells (mean aggregation)
      Cluster 3: 82 cells (mean aggregation)
      Cluster 4: 81 cells (mean aggregation)
      Cluster 5: 79 cells (mean aggregation)
      Cluster 6: 65 cells (mean aggregation)
      Cluster 7: 56 cells (mean aggregation)
      Cluster 8: 45 cells (mean aggregation)
      Cluster 9: 44 cells (mean aggregation)
      Cluster 10: 41 cells (mean aggregation)
      Cluster 11: 39 cells (mean aggregation)
      Cluster 12: 131 cells (mean aggregation)
      Cluster 13: 35 cells (mean aggregation)
   

In [1]:
# λ 越大更稀疏0.1-0.5
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --stage1_model_path "../deconv_results/PDAC/final_vae.pth" \
    --output_dir "../deconv_results/PDAC" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.1 \
    --loss_lambda_cosine 5 \
    --loss_lambda_pearson 5 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.1 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 400 \
    --weight_threshold 0.05

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample name: PDAC
Stage 1 model: ../deconv_results/PDAC/final_vae.pth
Output directory: ../deconv_results/PDAC
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 915 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 1926 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/PDAC/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([27, 256])
   Cluster expressions (marker): torch.Size([27, 915])
   Cluster expressions (all genes): 27 × 19738
   Loaded celltype mapping: 27 clusters
   Average cell counts: 3302.8
Loaded all genes list: 19738 genes
VAE Encoder loaded: 915 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '3', '4', '5', '6', '7', '8', '9']
Marker genes: 915
Using Stage 1 cluster centers and expressions...
Loaded 27 clusters
Using Stage 1 pretrained cluster data
   Cluste

GAT Training: 100%|██████████| 400/400 [05:11<00:00,  1.28epoch/s, Total=7.1378, Pearson=0.5360, MSE=22.3982, Cosine=0.4175, Proportion=1.2346]


Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 8560 -> 2002 (17.3%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/PDAC/PDAC_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Cancer', 'Ductal']. Merging corresponding cluster columns by summing weights.
   Columns before: 27, after merge: 11
   Saved cell composition (celltype): ../deconv_results/PDAC/PDAC_cell_composition.csv
   Saved cluster composition: ../deconv_results/PDAC/PDAC_cluster_composition.csv

Computing reconstruction quality per spot...
   Using 915 marker genes for reconstruction quality (consistent with training objective)
   Cosine similarities saved: ../deconv_results/PDAC/PDAC_spot_cosine_similarity.csv

Plotting reconstruction quality curve...
   Reconstruction quality curve saved: ../deconv_results/PDAC/PDAC_reconstruction_quality_curve.